In [115]:
from ts_toolkit.pipeline.runner import TsPipeline
from ts_toolkit.pipeline.utils import load_config
import logging
from tqdm.notebook import tqdm
import pandas as pd
from dotenv import load_dotenv
import os
import numpy as np
from gluonts.model.forecast import QuantileForecast
import utilsforecast.processing as ufp
from gluonts.model import Forecast
from multivar_timeseries.evaluation.metrics import MAR, MBR

# auto reload imports in jupyter
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [116]:
load_dotenv()
print(GIFT_EVAL_PATH := os.getenv("GIFT_EVAL"))
print(REPO_ROOT := os.getenv("REPO_ROOT"))

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

/home/roc/01_projects/01_timeseries/timeseries-research/data/gift_eval
/home/roc/01_projects/01_timeseries/timeseries-research


In [117]:
logging.basicConfig(level=logging.INFO)

# --- Load configuration ---
# config_path = "./configs/experiments/simpletime.yaml"
# config_path = "../configs/experiments/simpletime_sample.yaml"
config_path = "configs/experiments/personal/mar_mbr_debug.yaml"
# config_path = "./configs/experiments/gifteval.yaml"

config = load_config(config_path)

# --- Set up the evaluation pipeline ---
pipeline = TsPipeline(config)
pipeline.run(skip_existing=False)

pipeline.get_results()


INFO:root:   - Results will be saved to: /home/roc/01_projects/01_timeseries/timeseries-research/results/mar_mbr_debug_20251017-163941/ToTo-GiftEval-sample
INFO:root:   - Datasets will be loaded from: /home/roc/01_projects/01_timeseries/timeseries-research/data/gift_eval

INFO:root:
--- Starting Evaluation ---


Loading base config from <REPO_ROOT>/configs/experiments/personal/mar_mbr_debug.yaml...


Evaluating Datasets:   0%|          | 0/1 [00:00<?, ?it/s]


















100%|██████████| 16/16 [00:16<00:00,  1.06s/it]



100%|██████████| 976/976 [00:00<00:00, 3562.04it/s]


(512, 48) (512, 1)
(512, 48) (512, 1)


(512, 48) (512, 1)
(512, 48) (512, 1)


(512, 48) (512, 1)
(512, 48) (512, 1)


2000it [00:01, 1417.04it/s]
INFO:ts_toolkit.pipeline.eval:Results for bitbrains_rnd/H have been written to /home/roc/01_projects/01_timeseries/timeseries-research/results/mar_mbr_debug_20251017-163941/ToTo-GiftEval-sample/all_results.csv
INFO:ts_toolkit.pipeline.eval:Config has been written to /home/roc/01_projects/01_timeseries/timeseries-research/results/mar_mbr_debug_20251017-163941/ToTo-GiftEval-sample/mar_mbr_debug_20251017-163941_config.yaml
INFO:root:
--- ✅ All evaluations complete. ---


(464, 48) (464, 1)
(464, 48) (464, 1)


,dataset,variate,num_variates,model,eval_metrics/MAR[0.5][512],eval_metrics/MBR[0.5][512],eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss
0,bitbrains_rnd/H/short,UNI,2,ToTo-GiftEval-sample,1.212801,0.662227,1.875770e+06,1.875770e+06,139.199349,5.848582,0.833639,0.528964,202.245643,1369.587463,5.993985,0.609204,0.597404


In [4]:
config.experiment_name

'mar_mbr_debug_20251017-150940'

In [5]:
# import pandas as pd
# df_results = pd.read_csv(f"{REPO_ROOT}/results/{config.experiment_name}/{config.forecaster.alias}/all_results.csv")
# df_results

# Investigate

## Get Forecasts

In [6]:
import pandas as pd
import logging
from tqdm.notebook import tqdm
from ts_toolkit.models.ensembles.median import MedianEnsemble
from ts_toolkit.models import MULTIVARIATE_MODEL_LIST
from ts_toolkit.pipeline.gluonts_predictor import GluonTSPredictor
from ts_toolkit.pipeline.eval import Evaluator
from ts_toolkit.models import MODEL_REGISTRY, get_model
import os


In [7]:
dataset_cfg = pipeline.config.datasets[0]
to_univariate = pipeline.config.evaluation.to_univariate


In [8]:
evaluator = Evaluator(
    dataset_name=dataset_cfg.get("name"),
    term=dataset_cfg.get("term", "short"),
    to_univariate=to_univariate,
    # Create a subdirectory for each model's results
    output_path=f"{pipeline.output_path}/{pipeline.forecaster.alias}",
    storage_path=pipeline.storage_path,
    config=pipeline.config
)


In [9]:
dataset = evaluator.dataset.test_data.input
dataset

InputDataset(test_data=TestData(dataset=<gluonts.transform._base.TransformedDataset object at 0x722e26886b90>, splitter=OffsetSplitter(offset=-96), prediction_length=48, windows=2, distance=48, max_history=None))

In [10]:
h = pipeline.predictor.h or dataset.test_data.prediction_length
freq = pipeline.predictor.freq
print(h, freq, pipeline.predictor.batch_size)

48 None 1024


In [11]:
evaluator.forecasts = pipeline.predictor.predict(dataset)

100%|██████████| 976/976 [00:00<00:00, 3547.16it/s]


In [12]:
evaluator.forecasts.__len__(), [len(x.mean) for x in evaluator.forecasts][0]

(2000, 48)

In [13]:
batch = []
for _, entry in enumerate(dataset):
    batch.append(entry)

# So batch is w * n_var * T
len(batch), [wind['target'].shape for wind in batch]

(2000,
 [(624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (672,),
  (624,),
  (

In [14]:
# Extract all values for each variate across all windows
variates = batch[0]['target'].shape[0]  # number of variates
windows = len(batch)

# var_i: all values of variate i from all windows concatenated
var_dict = {}
for i in range(variates):
    var_dict[f'var_{i+1}'] = [batch[w]['target'][i].flatten() for w in range(windows)]
    var_dict[f'var_{i+1}'] = np.concatenate(var_dict[f'var_{i+1}'])

# var_i_j: all values of variate i from window j (j=1-based)
for i in range(variates):
    for j in range(windows):
        var_dict[f'var_{i+1}_{j+1}'] = batch[j]['target'][i].flatten()

# Example: var_1 contains all values of variate 1 from all windows
#          var_1_1 contains all values of variate 1 from window 1

# Check if the first 2376 values of all windows are the same for each variate
same_across_windows = {}
for i in range(variates):
    key = f'var_{i+1}_1'
    reference = var_dict[key][:2376]
    all_equal = True
    for j in range(2, windows+1):
        compare_key = f'var_{i+1}_{j}'
        if not np.array_equal(reference, var_dict[compare_key][:2376]):
            all_equal = False
            break
    same_across_windows[f'var_{i+1}'] = all_equal

same_across_windows

{'var_1': False,
 'var_2': False,
 'var_3': False,
 'var_4': False,
 'var_5': False,
 'var_6': False,
 'var_7': False,
 'var_8': False,
 'var_9': False,
 'var_10': False,
 'var_11': False,
 'var_12': False,
 'var_13': False,
 'var_14': False,
 'var_15': False,
 'var_16': False,
 'var_17': False,
 'var_18': False,
 'var_19': False,
 'var_20': False,
 'var_21': False,
 'var_22': False,
 'var_23': False,
 'var_24': False,
 'var_25': False,
 'var_26': False,
 'var_27': False,
 'var_28': False,
 'var_29': False,
 'var_30': False,
 'var_31': False,
 'var_32': False,
 'var_33': False,
 'var_34': False,
 'var_35': False,
 'var_36': False,
 'var_37': False,
 'var_38': False,
 'var_39': False,
 'var_40': False,
 'var_41': False,
 'var_42': False,
 'var_43': False,
 'var_44': False,
 'var_45': False,
 'var_46': False,
 'var_47': False,
 'var_48': False,
 'var_49': False,
 'var_50': False,
 'var_51': False,
 'var_52': False,
 'var_53': False,
 'var_54': False,
 'var_55': False,
 'var_56': False,
 

### Findings:
> 1. evaluator.dataset.test_data.input, which converts to the batch input, is in the shape of (n_windows, n_var, n_time_step), Each window and var contains all values of the previous window + 48 new time steps, which is the `h`
> 2. df is just all n_windows concatenated together vertically, each window with the same starting ds. So the number of rows of df = sum of all time steps in the (test) dataset. Multivariates are converted into y_i columns
> 3. predict_df takes df to run MedianEnsemble.forecast(), and returns fcst_df in the shape of (n_windows x n_var x h,  2 (unique_id, ds) + 1 (median) + 9 (quantiles)) or (6 x 7 x 48, 12)
> 4. Last bit of predict_df wrangles fcst_df to the shape of (n_windows x n_var, h) or (6 * 7, 48)

In [15]:
df = pipeline.predictor._gluonts_dataset_to_df(batch)
df.shape

(1024000, 3)

In [16]:
df

,unique_id,ds,y
0,rnd_142_dim0-2013-07-31 14:00,2013-07-05 14:00:00,0.000000
1,rnd_142_dim0-2013-07-31 14:00,2013-07-05 15:00:00,0.000000
2,rnd_142_dim0-2013-07-31 14:00,2013-07-05 16:00:00,0.000000
3,rnd_142_dim0-2013-07-31 14:00,2013-07-05 17:00:00,0.000000
4,rnd_142_dim0-2013-07-31 14:00,2013-07-05 18:00:00,0.000000
...,...,...,...
1023995,rnd_239_dim1-2013-08-04 14:00,2013-07-28 17:00:00,0.150000
1023996,rnd_239_dim1-2013-08-04 14:00,2013-07-28 18:00:00,0.183333
1023997,rnd_239_dim1-2013-08-04 14:00,2013-07-28 19:00:00,0.366667
1023998,rnd_239_dim1-2013-08-04 14:00,2013-07-28 20:00:00,0.172222


In [17]:
metadata = pipeline.predictor.multivar_metadata if len(pipeline.predictor.target_cols) > 1 else pipeline.predictor.univar_metadata

In [18]:
fcst_df = pipeline.predictor.forecaster.forecast(
            df=df,
            h=h,
            freq=freq,
            level=pipeline.predictor.level,
            quantiles=pipeline.predictor.quantiles,
        )

100%|██████████| 32/32 [00:35<00:00,  1.10s/it]


In [19]:
fcst_df

,unique_id,ds,ToTo-GiftEval-sample,ToTo-GiftEval-sample-q-10,ToTo-GiftEval-sample-q-20,ToTo-GiftEval-sample-q-30,ToTo-GiftEval-sample-q-40,ToTo-GiftEval-sample-q-50,ToTo-GiftEval-sample-q-60,ToTo-GiftEval-sample-q-70,ToTo-GiftEval-sample-q-80,ToTo-GiftEval-sample-q-90
0,rnd_100_dim0-2013-07-31 14:00,2013-07-26 22:00:00,89.469902,87.422112,87.933128,88.257843,88.646385,89.469902,90.537712,92.677216,97.442398,100.095871
1,rnd_100_dim0-2013-07-31 14:00,2013-07-26 23:00:00,87.266159,86.020462,86.660004,86.881676,87.030373,87.266159,87.441971,87.640526,88.317696,89.066299
2,rnd_100_dim0-2013-07-31 14:00,2013-07-27 00:00:00,88.566696,86.552887,87.527397,87.852547,88.094872,88.566696,89.915871,93.020126,98.291939,102.494278
3,rnd_100_dim0-2013-07-31 14:00,2013-07-27 01:00:00,87.641479,86.564636,86.849228,87.087227,87.259224,87.641479,87.983246,88.307991,89.111038,93.420670
4,rnd_100_dim0-2013-07-31 14:00,2013-07-27 02:00:00,105.752548,91.578728,97.060844,100.392235,103.436203,105.752548,109.211990,117.262535,122.315674,133.137283
...,...,...,...,...,...,...,...,...,...,...,...,...
95995,rnd_9_dim1-2013-08-04 14:00,2013-07-30 17:00:00,23.485645,23.258801,23.373310,23.376623,23.383026,23.485645,23.497286,23.602465,23.628244,23.720865
95996,rnd_9_dim1-2013-08-04 14:00,2013-07-30 18:00:00,23.494070,23.367195,23.373180,23.376301,23.468981,23.494070,23.499083,23.516602,23.617720,23.713125
95997,rnd_9_dim1-2013-08-04 14:00,2013-07-30 19:00:00,23.494429,23.366240,23.374407,23.379955,23.467733,23.494429,23.502792,23.599627,23.618780,23.775335
95998,rnd_9_dim1-2013-08-04 14:00,2013-07-30 20:00:00,23.494503,23.264637,23.374935,23.379745,23.481876,23.494503,23.500795,23.523710,23.617599,23.735132


In [20]:
fcst_df.shape

(96000, 12)

In [21]:
2016/48

42.0

In [22]:
fcst_df = fcst_df.set_index("unique_id")
fcsts: list[Forecast] = []
for uid, metadata_uid in tqdm(metadata.items(), total=len(metadata)):
    fcst_df_uid = fcst_df.loc[uid]
    forecast_arrays = ufp.value_cols_to_numpy(
        df=fcst_df_uid,
        id_col="unique_id",
        time_col="ds",
        target_col=None,
    )
    forecast_keys = ["mean"]
    if pipeline.predictor.quantiles is not None:
        forecast_keys += [f"{q}" for q in pipeline.predictor.quantiles]
    q_fcst = QuantileForecast(
        forecast_arrays=forecast_arrays.T,
        forecast_keys=forecast_keys,
        item_id=metadata_uid["item_id"],
        start_date=metadata_uid["fcst_start"],
    )
    fcsts.append(q_fcst)


  0%|          | 0/2000 [00:00<?, ?it/s]

In [23]:
fcsts.__len__(), fcsts[0].quantile(0.5).__len__()

(2000, 48)

In [24]:
fcsts_50qtl = [fcst.quantile(0.5) for fcst in fcsts]

In [25]:
# fcsts_arr = np.stack(fcsts)  # shape: (42, 48)

# # Reshape to (6, 7, h)
# fcsts_reshaped = fcsts_arr.reshape(6, 7, )
# fcsts_reshaped.shape

In [26]:
# fcsts_reshaped[0][0].quantile(0.5).shape

## Evaluate

### Evaluation Pipeline Summary

1. **Batching Data and Forecasts**  
   The original data and forecasts are passed to `gluonts.model.evaluation.evaluate_forecasts_raw`, where `data.input`, `data.label`, and forecasts are parsed and grouped into batches using a batcher.

2. **Batch Wrapping with ChainMap**  
   For each batch, the input, label, and forecasts are combined into a `ChainMap` object called `data_batch`, which is basically a merged dictionary for convenient access and is then passed to each metric (e.g., MAE, MSE) for evaluation. Here is why we need to modify the source code of `gluonts`, because in the original package `input` is only used to derive seasonality. The original `input` time series are not included in the final `data_batch`. 

3. **Metric Calculation Process**  
   - Each metric is instantiated as an evaluator, and the calculation is performed via `evaluator.update(data_batch)`.
   - The evaluator has two main components:
     - `stat`: Defines the error calculation (e.g., `absolute_error` for MAE). A `partial` wrapper is used to fix parameters such as `forecast_type='0.5'`.
     - `aggregate`: Typically a `Mean()` class with a specified axis for aggregation.
   - Under the hood, `evaluator.update` executes `aggregate.step(stat(data))`:
     - `stat(data)` computes the error for the batch (e.g., `absolute_error(data_batch, forecast_type='0.5')`).
     - `aggregate.step()` processes the error and updates:
       - `aggregate.partial_result`: The sum of all errors in the batch.
       - `aggregate.get()`: The mean error, calculated as `aggregate.partial_result / aggregate.n`.

4. **Aggregating Results Across Batches**  
   As the evaluation iterates through all batches, the evaluator accumulates the total error (`partial_result`) and the total number of time steps (`n`). The final average error across all batches is obtained via `aggregate.get()`.

### Notes
1. For multivar data, metrics are calculated with all vars flattened into one - aggregated over all time steps. Make sense when the number of steps among vars are largely different

### Preproc

In [27]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)
from multivar_timeseries.evaluation.metrics import MAR, MBR, mean_abs_deviation_label, mean_abs_deviation_forecast, _get_input_means
from gluonts.ev.metrics import absolute_error
from gluonts.itertools import batcher
from gluonts.ev.aggregations import Mean

from gluonts.model.evaluation import _get_data_batch
from toolz import first

In [28]:
test_data = evaluator.dataset.test_data
# test_data = evaluator.dataset.univar_test_data
axis = None
metrics = [
    # MAE(),
    MAR(context_length=512)
]

batch_size=512 # ?

# forecasts = fcsts_reshaped
forecasts = fcsts


EVALUATOR = Evaluator(
    dataset_name=dataset_cfg.get("name"),
    term=dataset_cfg.get("term", "short"),
    to_univariate=to_univariate,
    # Create a subdirectory for each model's results
    output_path=f"{pipeline.output_path}/{pipeline.forecaster.alias}",
    storage_path=pipeline.storage_path,
    config=pipeline.config
)

seasonality = EVALUATOR.seasonality
mask_invalid_label = True
allow_nan_forecast = False

In [29]:
label_ndim = first(test_data.label)["target"].ndim

assert label_ndim in [1, 2]

if axis is None:
    axis = tuple(range(label_ndim + 1))
if isinstance(axis, int):
    axis = (axis,)

assert all(ax in range(3) for ax in axis)

evaluators = {}
evaluator_metric = MAE()(axis=axis)
evaluators[evaluator_metric.name] = evaluator_metric


index_data = []

# TODO: Here the returned batch size is actually just one, within which there are several windows. 
# We may want to align the batcher and the windows, or remove batcher altogether.
input_batches = batcher(test_data.input, batch_size=batch_size)
label_batches = batcher(test_data.label, batch_size=batch_size)
forecast_batches = batcher(forecasts, batch_size=batch_size)

# pbar = tqdm()

input_batch, label_batch, forecast_batch = first(zip(input_batches, label_batches, forecast_batches))

if 0 not in axis:
    index_data.extend(
        [
            (forecast.item_id, forecast.start_date)
            for forecast in forecast_batch
        ]
    )

import logging
from collections import ChainMap
from typing import Iterable, List, Optional, Union
from dataclasses import dataclass
from toolz import first, valmap

import numpy as np
import pandas as pd
from tqdm import tqdm

from gluonts.dataset import DataEntry
from gluonts.dataset.split import TestData
from gluonts.ev.ts_stats import seasonal_error
from gluonts.itertools import batcher, prod
from gluonts.model import Forecast, Predictor
from gluonts.time_feature.seasonality import get_seasonality

from gluonts.model.evaluation import BatchForecast


data_batch = _get_data_batch(
            input_batch,
            label_batch,
            forecast_batch,
            seasonality=seasonality,
            mask_invalid_label=mask_invalid_label,
            allow_nan_forecast=allow_nan_forecast,
        )

### Metric Update

In [30]:
data_batch['label'].data.shape, data_batch['input'].__len__(), data_batch['input'][0].shape

((512, 48), 512, (624,))

In [31]:
mar = MAR(context_length=512)(axis=axis)
mar.update(data_batch)
mar.get()

(512, 48) (512, 1)
(512, 48) (512, 1)


1.2190791133512566

In [32]:
mae = MAE()(axis=axis)
mae.update(data_batch)
mae.get()

169.89621339087776

In [58]:
error_mae = absolute_error(data_batch, '0.5')
error_mar = mean_abs_deviation_forecast(data_batch, '0.5', 512)
aggregate_mae = Mean(axis=axis)
aggregate_mar = Mean(axis=axis)
aggregate_mae.step(error_mae), aggregate_mae.partial_result, aggregate_mar.step(error_mar), aggregate_mar.partial_result

(None, array(4116107.65210631), None, array(nan))

In [59]:
error_mae.shape, error_mar.shape

((512, 48), (512, 48))

In [60]:
error_mae[240]

masked_array(data=[--, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --],
             mask=[ True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True],
       fill_value=1e+20,
            dtype=float64)

In [64]:
error_mar

array([[15.66883132, 15.5768666 , 15.72213027, ..., 15.40877578,
        15.63030288, 16.18305251],
       [12.48391211, 12.49450171, 12.48482764, ..., 12.57032264,
        12.8550669 , 12.57158912],
       [ 0.15033649,  0.1505035 ,  0.15196357, ...,  0.15336679,
         0.14943157,  0.15235136],
       ...,
       [ 2.06478488,  2.20014178,  2.19798648, ...,  2.19234263,
         2.18658625,  2.17609393],
       [ 0.09352351,  0.09625436,  0.09551192, ...,  0.11128677,
         0.11866135,  0.12029619],
       [ 0.07568495,  0.07008045,  0.0739172 , ...,  0.07414549,
         0.08382075,  0.07350193]])

In [65]:
axis

(0, 1)

In [66]:
# For MAE (masked array)
isnan_mae_raw = np.isnan(error_mae.data)                # raw NaN locations
mask_mae = np.ma.getmaskarray(error_mae)                # mask positions
isnan_mae_unmasked = isnan_mae_raw & ~mask_mae          # NaNs that are NOT masked

print("NaNs in MAE raw:", isnan_mae_raw.any())
print("NaNs in MAE unmasked:", isnan_mae_unmasked.any())
print("np.isnan(error_mae).any() (mask-aware):", np.isnan(error_mae).any())

# For MAR (plain ndarray)
isnan_mar = np.isnan(error_mar)
print("NaNs in MAR:", isnan_mar.any())

NaNs in MAE raw: True
NaNs in MAE unmasked: False
np.isnan(error_mae).any() (mask-aware): False
NaNs in MAR: True


In [68]:
mask_mae

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [67]:
# For MAE (masked array)
isnan_mae_raw = np.isnan(error_mar.data)                # raw NaN locations
mask_mae = np.ma.getmaskarray(error_mar)                # mask positions
isnan_mae_unmasked = isnan_mae_raw & ~mask_mae          # NaNs that are NOT masked

print("NaNs in MAE raw:", isnan_mae_raw.any())
print("NaNs in MAE unmasked:", isnan_mae_unmasked.any())
print("np.isnan(error_mae).any() (mask-aware):", np.isnan(error_mar).any())


NaNs in MAE raw: True
NaNs in MAE unmasked: True
np.isnan(error_mae).any() (mask-aware): True


In [44]:
type(error_mar)

numpy.ma.core.MaskedArray

In [45]:
# NaNs that are not masked
nan_unmasked_mae = np.isnan(error_mae.data) & ~np.ma.getmaskarray(error_mae)
nan_unmasked_mar = np.isnan(error_mar.data) & ~np.ma.getmaskarray(error_mar)

nan_unmasked_mae.any(), nan_unmasked_mar.any()

(False, False)

In [46]:
error_mar.data[240]

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan])

In [73]:
np.ma.masked_invalid(error_mar)

masked_array(
  data=[[15.668831320369947, 15.576866598690259, 15.722130270565259, ...,
         15.408775778377759, 15.630302877987134, 16.183052511776197],
        [12.483912114496832, 12.494501714106207, 12.484827641840582, ...,
         12.57032263695777, 12.855066899653082, 12.571589116449957],
        [0.15033648695264534, 0.150503499167306, 0.15196357454572396,
         ..., 0.1533667870930262, 0.14943156923566536,
         0.15235136236463265],
        ...,
        [2.064784877033155, 2.200141780109327, 2.1979864761542487, ...,
         2.1923426315497565, 2.1865862533759284, 2.176093928546827],
        [0.09352351026416317, 0.09625435666919246, 0.0955119212138702,
         ..., 0.11128676729083553, 0.11866135196567074,
         0.12029618816257015],
        [0.0756849545166205, 0.07008044897768007, 0.07391719996187196,
         ..., 0.0741454857513617, 0.08382075010988221,
         0.07350193440172181]],
  mask=[[False, False, False, ..., False, False, False],
        [False, 

In [48]:
np.all(np.ma.getmaskarray(error_mae) == np.ma.getmaskarray(error_mar))

True

In [49]:
mask = np.ma.getmaskarray(error_mar)  # == mask of error_mar_1

unmasked_nan_mae = np.isnan(error_mae.data) & ~mask
unmasked_nan_mar = np.isnan(error_mar.data) & ~mask

unmasked_nan_mae.any(), unmasked_nan_mar.any()
# np.where(unmasked_nan_mar & ~unmasked_nan_mae)  # positions only MAR has as unmasked-NaN

(False, False)

In [50]:
all_na_inputs = set()
for i in range(len(unmasked_nan_mar)):
    for j in range(len(unmasked_nan_mar[i])):
        if unmasked_nan_mar[i][j]:
            all_na_inputs.add(i)
print(all_na_inputs)

set()


In [51]:
data_batch["input"][480]

masked_array(data=[--, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --

In [52]:
data_batch["label"][480]

masked_array(data=[--, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --, --, --, --, --, --, --, --, --,
                   --, --, --, --, --, --],
             mask=[ True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True,  True,  True,  True,  True,  True],
       fill_value=1e+20,
            dtype=float32)

In [53]:
data_batch['0.5'][480]

array([ 5.50045625e-05,  1.00256686e-04, -1.34012895e-04, -7.13288464e-05,
        5.84390473e-05,  7.54048524e-05,  3.66014319e-06,  2.12181185e-04,
        2.31261685e-04,  3.04113601e-05, -1.30663422e-04, -3.20881481e-05,
       -9.89199179e-05, -4.89374224e-06, -2.31683807e-04, -4.19094031e-05,
       -9.01974709e-05,  1.92381427e-04,  2.57126339e-05,  6.88633736e-05,
        1.90823674e-04, -1.67208040e-04, -4.94564374e-05,  7.02530961e-05,
        9.51041238e-06, -7.57214875e-05, -3.89897396e-05, -4.03804588e-05,
        2.15299806e-05,  1.37789641e-04,  1.30922068e-04, -8.36362015e-05,
        1.68273327e-04,  3.21259286e-05, -1.52105466e-04,  2.00964307e-04,
       -1.70201874e-05, -8.23793671e-05, -8.14106388e-05,  1.81493378e-04,
        7.97296161e-05, -1.05404733e-05, -7.69768958e-05,  2.24206997e-05,
        3.00228712e-05,  1.19985118e-04, -8.54643222e-05,  9.41466133e-05])

In [54]:
data_batch["input"][479]

masked_array(data=[0.5166666507720947, 0.5611110925674438,
                   0.5333333611488342, 0.5333333611488342,
                   0.5222222208976746, 0.5666666626930237,
                   0.5333333611488342, 0.5388888716697693,
                   0.5611110925674438, 0.5333333611488342,
                   0.5444444417953491, 0.5444444417953491,
                   0.5333333611488342, 0.5111111402511597,
                   0.5277777910232544, 0.5333333611488342,
                   0.5333333611488342, 0.5277777910232544,
                   0.5333333611488342, 0.5333333611488342,
                   0.5388888716697693, 0.5444444417953491,
                   0.5388888716697693, 0.5458333492279053,
                   0.5166666507720947, 0.5444444417953491,
                   0.5333333611488342, 0.5444444417953491,
                   0.5444444417953491, 0.550000011920929,
                   0.5277777910232544, 0.5444444417953491,
                   0.550000011920929, 0.5277777910232544,

In [55]:
data_batch["label"][479]

masked_array(data=[0.13333334028720856, 0.10000000149011612,
                   0.13333334028720856, 0.10000000149011612,
                   0.10000000149011612, 0.12222222238779068,
                   0.10555555671453476, 0.1111111119389534,
                   0.10000000149011612, 0.09444444626569748,
                   0.13333334028720856, 0.13333334028720856,
                   0.10000000149011612, 0.13333334028720856,
                   0.10000000149011612, 0.10000000149011612,
                   0.13333334028720856, 0.13333334028720856,
                   0.10000000149011612, 0.10000000149011612,
                   0.13333334028720856, 0.10000000149011612,
                   0.10000000149011612, 0.10000000149011612,
                   0.13333334028720856, 0.10000000149011612,
                   0.10555555671453476, 0.12777778506278992,
                   0.10000000149011612, 0.13333334028720856,
                   0.10000000149011612, 0.10000000149011612,
                   0.1000

In [67]:
from gluonts.ev.stats import absolute_error

from gluonts.ev.metrics import DirectMetric
from gluonts.ev.aggregations import Mean
from functools import partial


In [126]:
# partial allows you to fix a certain number of arguments of a function and generate a new function.
# Here, we are fixing the forecast_type argument of absolute_error to '0.5'
stat = partial(absolute_error, forecast_type='0.5')
aggregate = Mean(axis=axis)

In [127]:
mae = DirectMetric(
            name=f"MAE['0.5']",
            stat=stat,
            aggregate=aggregate,
        )

In [128]:
error_batch = stat(data_batch)
# equivalent to: error_batch = np.abs(data_batch['label'] - data_batch['0.5'])

In [129]:
stat(data_batch) == absolute_error(data_batch, forecast_type='0.5')

masked_array(
  data=[[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]],
  mask=False,
  fill_value=True)

In [130]:
aggregate.step(error_batch)

In [131]:
print(aggregate.partial_result == sum(sum(error_batch.data)))
print(aggregate.get() == aggregate.partial_result / aggregate.n)

True
True


In [132]:
aggregate.n

array(2016.)

In [133]:
2016/42

48.0

In [124]:
evaluator_metric.update(data_batch)

DirectMetric(name='MAE[0.5]', stat=functools.partial(<function absolute_error at 0x7651374e6d40>, forecast_type='0.5'), aggregate=Mean(axis=(0, 1), partial_result=array(71368.07279547), n=array(14112.)))

In [125]:
evaluator_metric.aggregate.partial_result, evaluator_metric.aggregate.n, evaluator_metric.aggregate.get()

(array(71368.07279547), array(14112.), 5.0572613942370355)

In [114]:
print(evaluator_metric.get() == aggregate.get())

True


### MBR Investigation

In [97]:
from multivar_timeseries.evaluation.metrics import mean_bias
def count_nan(data_with_mask):
    return data_with_mask.mask.flatten().sum()

In [34]:
mbr = MBR(context_length=512)(axis=axis)
mbr.update(data_batch)
mbr.get()

0.6801580698835274

In [110]:
input_means = np.ma.masked_invalid(_get_input_means(data_batch, context_length=512))

In [112]:
error_mbr = mean_bias(data_batch, '0.5', context_length=512)

In [35]:
mbr.aggregate.partial_result, mbr.aggregate.n, mbr.aggregate.get()

(array(16351.), array(24040.), 0.6801580698835274)

In [74]:
512 * 48 - 24040

536

In [114]:
error_mbr[481]

masked_array(data=[--, --, --, --, --, --, --, --, --, --, --, False,
                   False, False, False, False, False, False, False, False,
                   False, False, False, False, False, False, False, False,
                   False, False, False, False, False, False, False, False,
                   False, False, False, False, False, False, False, False,
                   False, False, False, False],
             mask=[ True,  True,  True,  True,  True,  True,  True,  True,
                    True,  True,  True, False, False, False, False, False,
                   False, False, False, False, False, False, False, False,
                   False, False, False, False, False, False, False, False,
                   False, False, False, False, False, False, False, False,
                   False, False, False, False, False, False, False, False],
       fill_value=True)

In [101]:
a = (data_batch["label"] - input_means).data.flatten()
nan_count = np.isnan(a).sum()
print(nan_count)

b = (data_batch["label"] - data_batch['0.5']).data.flatten()
nan_count = np.isnan(b).sum()
print(nan_count)

536
536


In [ ]:
np.i(data_batch["label"].data.flatten()).sum()

536

In [ ]:
data_batch["label"][481]

masked

In [69]:
(data_batch["label"] - data_batch['0.5'])

masked_array(
  data=[[-1.0530853271484375, -0.101654052734375, 0.2161865234375, ...,
         0.222564697265625, -0.2225494384765625, 2.0410003662109375],
        [0.577850341796875, 0.1576385498046875, 0.7340087890625, ...,
         32.10844421386719, 35.20161437988281, 33.619720458984375],
        [-0.01272130012512207, -0.0018885135650634766,
         -0.00020420551300048828, ..., -0.003792285919189453,
         0.0006341934204101562, 0.01824498176574707],
        ...,
        [-0.2426128387451172, 0.5671882629394531, -0.16798973083496094,
         ..., -1.6316490173339844, -1.4459495544433594,
         -1.1308841705322266],
        [0.001775205135345459, 0.030570924282073975,
         0.0054628849029541016, ..., 0.05459582805633545,
         0.0357896089553833, 0.02800661325454712],
        [-0.008199572563171387, 0.020541608333587646,
         -0.006659567356109619, ..., -0.05645543336868286,
         -0.05241954326629639, -0.03852510452270508]],
  mask=[[False, False, False, ...

768

In [65]:
c = np.sign(a) == np.sign(b)

In [57]:
b = mean_bias(data_batch, '0.5').data.flatten()

In [58]:
np.isnan(b).any()

False

In [72]:
np.nanmean(data_batch["input"][0], axis=0, keepdims=True)[0]

147.83777

In [68]:
data_batch["input"][0].mean()

nan

In [54]:
np.sign(data_batch["label"] - input_means) == np.sign(data_batch["label"] - data_batch['0.5'])

masked_array(
  data=[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],
  mask=[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],
  fill_value=True)

### MAR Investigation

In [75]:
mad_label = mean_abs_deviation_label(data_batch, 512)
mad_fcst = mean_abs_deviation_forecast(data_batch, '0.5', 512)

(512, 48) (512, 1)
(512, 48) (512, 1)


In [96]:
input_means

array([[1.47837769e+02],
       [1.50578857e+02],
       [1.42151749e+00],
       [1.44787407e+00],
       [2.62990742e+01],
       [1.99242668e+01],
       [8.98806751e-01],
       [6.80938840e-01],
       [1.39595165e+01],
       [1.39714632e+01],
       [2.38542706e-01],
       [2.38746867e-01],
       [4.74950000e+03],
       [4.19023975e+03],
       [4.05801468e+01],
       [3.58017845e+01],
       [1.59743896e+02],
       [1.59000351e+02],
       [1.36486602e+00],
       [1.35851324e+00],
       [3.44673653e+01],
       [3.49478493e+01],
       [1.17796886e+00],
       [1.19439006e+00],
       [5.69609451e+01],
       [5.68891411e+01],
       [1.94671762e+00],
       [1.94426346e+00],
       [2.61275269e+03],
       [2.61803857e+03],
       [5.02452583e+01],
       [5.03469162e+01],
       [5.92489100e+00],
       [5.80866003e+00],
       [2.02491209e-01],
       [1.98518842e-01],
       [4.17096634e+01],
       [4.13130569e+01],
       [1.42548454e+00],
       [1.41192997e+00],


In [78]:
count_nan(mad_label), count_nan(mad_fcst)

(768, 768)

In [95]:
count_nan(data_batch['label']), count_nan(input_means)

AttributeError: 'numpy.ndarray' object has no attribute 'mask'

In [ ]:
count_nan(np.abs(data_batch["label"] - input_means_masked))

768